In [ ]:
from openpyxl import load_workbook
import pandas as pd
from collections import defaultdict

In [ ]:
def expandir_mescladas(ws):
    """
    Copia o valor da célula superior esquerda para todas as células
    pertencentes à região mesclada.
    """
    for merged in list(ws.merged_cells.ranges):

        valor = ws.cell(
            merged.min_row,
            merged.min_col
        ).value

        min_row = merged.min_row
        max_row = merged.max_row
        min_col = merged.min_col
        max_col = merged.max_col

        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(r, c).value = valor

In [ ]:
def montar_cabecalho(ws):
    """
    Monta automaticamente o cabeçalho da planilha PPI.

    Expande células mescladas e propaga os títulos horizontal e
    verticalmente antes de concatenar os níveis do cabeçalho.
    """

    # Expande as células mescladas
    expandir_mescladas(ws)

    # Linhas do cabeçalho (Excel)
    HEADER_ROWS = [2, 3, 4, 5, 6, 7]

    # ------------------------------------------------------------------
    # Cria uma matriz contendo somente o cabeçalho
    # ------------------------------------------------------------------

    cab = []

    for r in HEADER_ROWS:

        linha = []

        for c in range(1, ws.max_column + 1):

            linha.append(ws.cell(r, c).value)

        cab.append(linha)

    cab = pd.DataFrame(cab)

    # ------------------------------------------------------------------
    # Propaga valores das células mescladas
    # ------------------------------------------------------------------

    # esquerda -> direita
    cab = cab.ffill(axis=1)

    # cima -> baixo
    cab = cab.ffill(axis=0)

    # ------------------------------------------------------------------
    # Limpa valores inúteis
    # ------------------------------------------------------------------

    lixo = {
        "#REF!",
        "TOTAL",
        "SOMA",
        0,
        "0",
        None,
        ""
    }

    cab = cab.replace(list(lixo), pd.NA)

    # ------------------------------------------------------------------
    # Monta o nome de cada coluna
    # ------------------------------------------------------------------

    colunas = []

    for c in range(cab.shape[1]):

        partes = []

        for r in range(cab.shape[0]):

            valor = cab.iat[r, c]

            if pd.isna(valor):
                continue

            valor = str(valor).strip()

            if valor == "":
                continue

            # Ignora fórmulas
            if valor.startswith("="):
                continue

            # Ignora erros do Excel
            if valor.startswith("#"):
                continue

            # Ignora TOTAL e SOMA
            if valor.upper() in ("TOTAL", "SOMA"):
                continue

            # Ignora números puros
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass

            # =============================================================
            # ✅ CORREÇÃO: aqui é onde o valor válido é adicionado à lista
            # =============================================================
            partes.append(valor)

        colunas.append(" - ".join(partes))

    # ------------------------------------------------------------------
    # Remove duplicados
    # ------------------------------------------------------------------

    contador = defaultdict(int)

    novas = []

    for nome in colunas:

        contador[nome] += 1

        if contador[nome] == 1:

            novas.append(nome)

        else:

            novas.append(f"{nome}__{contador[nome]}")

    return novas

In [ ]:
ARQUIVO = "Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - 2027.xlsx"

wb = load_workbook(
    ARQUIVO,
    data_only=False
)

ws = wb["META(2027)"]

colunas = montar_cabecalho(ws)

In [ ]:
dados = list(ws.values)

df = pd.DataFrame(
    dados[7:],   # linha 8 do Excel = índice 7
    columns=colunas
)

In [ ]:
for i, c in enumerate(colunas):

    if "PAVIMENTAÇÃO" in c.upper():
        print(i, c)

In [ ]:
for i, c in enumerate(colunas):

    if "PROCESSO" in c.upper():
        print(i, c)

In [ ]:
indice_colunas = pd.DataFrame({
    "indice": range(len(df.columns)),
    "coluna": df.columns
})

display(indice_colunas)

In [ ]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "PROCESSO",
        case=False,
        na=False
    )
]

In [ ]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "LICEN",
        case=False,
        na=False
    )
]

In [ ]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "FINANCEIRO PLAN",
        case=False,
        na=False
    )
]

In [ ]:
df.columns = [
    str(c)
        .replace("\n", " ")
        .replace("\r", " ")
        .strip()
    for c in df.columns
]

In [ ]:
print("Total de colunas:", len(df.columns))
print("Colunas únicas:", len(set(df.columns)))

In [ ]:
import pandas as pd

cab = []

for c in range(1, ws.max_column + 1):

    info = {
        "coluna": c
    }

    for r in range(1, 9):

        info[f"L{r}"] = ws.cell(r, c).value

    cab.append(info)

cab = pd.DataFrame(cab)

cab.head()

In [ ]:
def limpar(x):

    if x is None:
        return ""

    x = str(x).strip()

    if x.startswith("="):
        return ""

    if x == "#REF!":
        return ""

    return x

for c in cab.columns[1:]:

    cab[c] = cab[c].apply(limpar)

In [ ]:
pd.set_option("display.max_colwidth", None)

cab.iloc[175:205]

In [ ]:
# %% =====================================================================
# CRIAÇÃO DO DATAFRAME FINAL COM AS 43 COLUNAS ESPECIFICADAS
# ========================================================================

# Lista atualizada com as novas colunas
colunas_alvo = [
    "ID-ÚNICO",
    "SETOR",
    "UF",
    "BR",
    "EMPREENDIMENTO",
    "EXECUTOR (Grupo Controlador)",
    "DATA DE INÍCIO DA CONCESSÃO",
    "ANO DA CONCESSÃO",
    "INTERFERÊNCIAS",
    "INTERVENÇÃO AFETADA",
    "Descrição",
    "km(i)",
    "km(f)",
    "Ext.(km)",
    "Ext.(km)!",
    "FINANCEIRO(R$)",
    "FINANCEIRO(R$)!",
    "PER ANO(Execução)",
    "LP",
    "LI",
    "LO",
    "DATA DE INÍCIO",
    "SINAFLOR",
    "SISGLAF",
    "SEI",
    "Risco Principal",
    "CRITICIDADE",
    "SITUAÇÃO",
    "COMENTÁRIOS GERAIS",
    "Data do Atendimento",
    "NÍVEL DE ALERTA",
    "Emitida",
    "Em análise",
    "Aguardando",
    "Solicitada",
    "Sem informação",
    "Pendência",
    "Outras Descrições",
    # ===== NOVAS COLUNAS ADICIONADAS =====
    "Nº PROCESSO",
    "DATA DO PROTOCOLO",
    "LICENÇA",
    "SITUAÇÃO GERAL",
    # ====================================
    "PLANILHA"
]

# Cria o DataFrame final vazio, com o mesmo índice do df original
df_final = pd.DataFrame(index=df.index, columns=colunas_alvo)

# ========================================================================
# 1. DEFINIÇÃO DOS MAPEAMENTOS (palavras-chave para cada coluna destino)
# ========================================================================

mapeamento = {
    "ID-ÚNICO": ["ID", "UNICO"],
    "SETOR": ["SETOR"],
    "UF": ["UF"],
    "BR": ["BR"],
    "EMPREENDIMENTO": ["EMPREENDIMENTO"],
    "EXECUTOR (Grupo Controlador)": ["EXECUTOR", "CONTROLADOR"],
    "DATA DE INÍCIO DA CONCESSÃO": ["CONCESSÃO", "INÍCIO"],
    "ANO DA CONCESSÃO": ["ANO", "CONCESSÃO"],
    "INTERFERÊNCIAS": ["INTERFER"],
    "INTERVENÇÃO AFETADA": ["INTERVENÇÃO", "AFETADA"],
    "Descrição": ["DESCRIÇÃO"],
    "km(i)": ["KM I", "KM INICIAL"],
    "km(f)": ["KM F", "KM FINAL"],
    "Ext.(km)": ["EXT", "KM"],  # pega a primeira com EXT e KM
    "Ext.(km)!": ["EXT", "!", "KM"],  # pega a que tem "!"
    "FINANCEIRO(R$)": ["FINANCEIRO", "R$"],
    "FINANCEIRO(R$)!": ["FINANCEIRO", "R$", "!"],
    "PER ANO(Execução)": ["PER ANO", "EXECUÇÃO"],
    "LP": ["LP"],
    "LI": ["LI"],
    "LO": ["LO"],
    "DATA DE INÍCIO": ["DATA DE INÍCIO"],  # será tratado com exclusão de "CONCESSÃO"
    "SINAFLOR": ["SINAFLOR"],
    "SISGLAF": ["SISGLAF"],
    "SEI": ["SEI"],
    "Risco Principal": ["RISCO", "PRINCIPAL"],
    "CRITICIDADE": ["CRITICIDADE"],
    "SITUAÇÃO": ["SITUAÇÃO"],  # sem a palavra GERAL para não confundir
    "COMENTÁRIOS GERAIS": ["COMENTÁRIOS", "GERAIS"],
    "Data do Atendimento": ["ATENDIMENTO"],
    "NÍVEL DE ALERTA": ["ALERTA"],
    "Emitida": ["EMITIDA"],
    "Em análise": ["ANÁLISE"],
    "Aguardando": ["AGUARDANDO"],
    "Solicitada": ["SOLICITADA"],
    "Sem informação": ["SEM INFORMAÇÃO"],
    "Pendência": ["PENDÊNCIA"],
    "Outras Descrições": ["OUTRAS", "DESCRI"],
    # ===== MAPEAMENTO DAS NOVAS COLUNAS =====
    "Nº PROCESSO": ["PROCESSO", "Nº"],  # ou "NÚMERO" se preferir
    "DATA DO PROTOCOLO": ["PROTOCOLO", "DATA"],
    "LICENÇA": ["LICENÇA"],
    "SITUAÇÃO GERAL": ["SITUAÇÃO", "GERAL"],  # distinta de "SITUAÇÃO" sozinha
    # ========================================
    "PLANILHA": []  # será preenchida manualmente
}

# ========================================================================
# 2. FUNÇÃO PARA ENCONTRAR A COLUNA DE ORIGEM
# ========================================================================

def encontrar_coluna_origem(palavras, ignorar=None):
    """
    Retorna o nome da coluna em df.columns que contém TODAS as palavras da lista.
    Se 'ignorar' for passado, ignora colunas que contenham essa palavra.
    """
    for col in df.columns:
        col_str = str(col).upper()
        # Se tiver palavra para ignorar, pula
        if ignorar and ignorar.upper() in col_str:
            continue
        # Verifica se todas as palavras estão presentes
        if all(p.upper() in col_str for p in palavras):
            return col
    return None

# ========================================================================
# 3. PREENCHIMENTO COLUNA POR COLUNA
# ========================================================================

print("🔍 Mapeamento realizado:\n")

for destino, palavras in mapeamento.items():
    
    # Caso especial: PLANILHA
    if destino == "PLANILHA":
        df_final[destino] = "META (2027)"
        print(f"  {destino:30} <- (preenchido com 'META (2027)')")
        continue
    
    # Caso especial: "DATA DE INÍCIO" (da intervenção) deve ignorar "CONCESSÃO"
    if destino == "DATA DE INÍCIO":
        origem = encontrar_coluna_origem(palavras, ignorar="CONCESSÃO")
    else:
        origem = encontrar_coluna_origem(palavras)
    
    if origem:
        df_final[destino] = df[origem]
        print(f"  {destino:30} <- {origem}")
    else:
        print(f"  {destino:30} <- ⚠️ NÃO ENCONTRADO (deixado como NaN)")

# ========================================================================
# 4. (OPCIONAL) AJUSTES FINOS MANUAIS
# ========================================================================
# Se alguma coluna foi mapeada errada, você pode sobrescrever aqui.
# Exemplo:
# col_correta = encontrar_coluna_origem(["PROCESSO"], ignorar="Nº")  # ajuste conforme necessário
# if col_correta:
#     df_final["Nº PROCESSO"] = df[col_correta]

# ========================================================================
# 5. EXIBE E SALVA
# ========================================================================

print("\n📊 Primeiras linhas do DataFrame final:")
print(df_final.head())

print(f"\nFormato final: {df_final.shape[0]} linhas x {df_final.shape[1]} colunas")

# Salva em Excel
arquivo_final = "PPI_Colunas_Especificas - 2027.xlsx"
with pd.ExcelWriter(arquivo_final, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="META_2027")

print(f"\n✅ Arquivo exportado: {arquivo_final}")

In [ ]:
arquivo_saida = "PPI_Tratada - 2027.xlsx"

with pd.ExcelWriter(
    arquivo_saida,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False,
        sheet_name="META(2027)"
    )

print("Exportação concluída.")